In [10]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

In [11]:
df = pd.read_csv("../Data/Clean_Dataset.csv", index_col=0)
save_file_name = "Preprocessed_data"
print(df.columns)
df.head()

Index(['airline', 'flight', 'source_city', 'departure_time', 'stops',
       'arrival_time', 'destination_city', 'class', 'duration', 'days_left',
       'price'],
      dtype='object')


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


### (optional) Removing extreme prices

In [4]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)

IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

df = df[df["price"] <= upper_bound]
save_file_name = "Preprocessed_data_no_outliers"


### un flight (code) a le meme depart, destination et airline

In [12]:
df.drop(columns=["flight","days_left","price",'duration','class']).duplicated(keep="first").sum()

np.int64(296484)

In [13]:
df.drop(columns=["days_left","price",'duration','class']).duplicated(keep="first").sum()

np.int64(292513)

### donc on peut enlevé la colonne de flight puisqu'elle resume les autres colonnes

In [14]:
df = df.drop(columns=['flight'])

In [15]:
nominal_cols = ["airline","source_city","destination_city"]
ordinal_cols = ["departure_time","arrival_time","stops","class"]
numeric_cols = ["duration", "days_left"]
time_order = [ 'Early_Morning',    'Morning',    'Afternoon',    'Evening',    'Night',    'Late_Night']
stops_order = ['zero', 'one', 'two_or_more']
class_order = ['Economy', 'Business']
ordinal_categories = [ time_order, time_order, stops_order, class_order]

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_cols),
        ("ordinal", OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ("num", StandardScaler(), numeric_cols)
    ],
    verbose_feature_names_out=True
).set_output(transform='pandas')

X_encoded = preprocessor.fit_transform(df)

In [17]:
X_encoded.head()

,nominal__airline_AirAsia,nominal__airline_Air_India,nominal__airline_GO_FIRST,nominal__airline_Indigo,nominal__airline_SpiceJet,nominal__airline_Vistara,nominal__source_city_Bangalore,nominal__source_city_Chennai,nominal__source_city_Delhi,nominal__source_city_Hyderabad,...,nominal__destination_city_Delhi,nominal__destination_city_Hyderabad,nominal__destination_city_Kolkata,nominal__destination_city_Mumbai,ordinal__departure_time,ordinal__arrival_time,ordinal__stops,ordinal__class,num__duration,num__days_left
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,3.0,4.0,0.0,0.0,-1.397531,-1.843875
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,-1.375284,-1.843875
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.397531,-1.843875
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,-1.386407,-1.843875
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,-1.375284,-1.843875


### Saving the preprocessed data

In [18]:
import joblib

# Save everything
joblib.dump(X_encoded,"X_" + save_file_name + ".pkl")
joblib.dump(df['price'],"Y_" +save_file_name +"y.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
# To Load in another notebook
## df = pd.read_pickle("train_data.pkl")

['preprocessor.pkl']

### une seconde preprocessor avec different strategy (ordinal first)

In [19]:
# Define your column lists and categories
nominal_cols = ["airline", "source_city", "destination_city"]
ordinal_cols = ["departure_time", "arrival_time", "stops", "class"]
numeric_cols = ["duration", "days_left"]

time_order = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']
stops_order = ['zero', 'one', 'two_or_more']
class_order = ['Economy', 'Business']
ordinal_categories = [time_order, time_order, stops_order, class_order]

# The Modified ColumnTransformer
modified_preprocessor = ColumnTransformer(
    transformers=[
        # Auto-assign numeric values to nominal columns (e.g., Vistara=0, Air_India=1)
        ("nominal_encoded", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), nominal_cols),

        # Enforce your strict order for ordinal columns
        ("ordinal_encoded", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1), ordinal_cols),

        # Skip scaling, just pass the numbers through
        ("num", "passthrough", numeric_cols)
    ],
    verbose_feature_names_out=True
).set_output(transform='pandas')

modified_preprocessor.fit_transform(df).head()

,nominal_encoded__airline,nominal_encoded__source_city,nominal_encoded__destination_city,ordinal_encoded__departure_time,ordinal_encoded__arrival_time,ordinal_encoded__stops,ordinal_encoded__class,num__duration,num__days_left
0,4.0,2.0,5.0,3.0,4.0,0.0,0.0,2.17,1
1,4.0,2.0,5.0,0.0,1.0,0.0,0.0,2.33,1
2,0.0,2.0,5.0,0.0,0.0,0.0,0.0,2.17,1
3,5.0,2.0,5.0,1.0,2.0,0.0,0.0,2.25,1
4,5.0,2.0,5.0,1.0,1.0,0.0,0.0,2.33,1


In [20]:
joblib.dump(modified_preprocessor, "preprocessor2.pkl")

['preprocessor2.pkl']